# MathVerifyProject - Google Colab Version

This notebook provides a complete mathematical reasoning verification system in Google Colab.

**Features:**
- ✅ Mathematical expression verification
- ✅ LaTeX rendering
- ✅ Error classification
- ✅ Interactive Gradio web interface
- ✅ Sample problems from MATH-V

**Just run all cells to get started!**


## Step 1: Install Dependencies

This will install all required packages.


In [ ]:
# Install all dependencies
%pip install -q gradio>=4.0.0
%pip install -q math-verify[antlr4_13_2]
%pip install -q latex2sympy2_extended==1.10.2
%pip install -q antlr4-python3-runtime==4.13.2
%pip install -q sympy
%pip install -q rich
%pip install -q tqdm

print("✅ All dependencies installed!")


## Step 2: Setup System

Clone repositories and set up the system.


In [ ]:
import sys
import os

# Create project structure
os.makedirs("MathVerifyProject", exist_ok=True)
os.chdir("MathVerifyProject")

# Clone repositories (if not already present)
repos = {
    "Math-Verify": "https://github.com/huggingface/Math-Verify.git",
    "MATH-V": "https://github.com/mathllm/MATH-V.git"
}

import subprocess

for repo_name, repo_url in repos.items():
    if not os.path.exists(repo_name):
        print(f"Cloning {repo_name}...")
        subprocess.run(["git", "clone", "-q", repo_url], check=False)
    else:
        print(f"✓ {repo_name} already exists")

print("\n✅ Setup complete!")


## Step 3: Create Core Verification Module


In [ ]:
# Create core verification module
import sys
sys.path.insert(0, 'Math-Verify/src')

from math_verify import parse, verify, ExprExtractionConfig, LatexExtractionConfig
import warnings

class MathVerifier:
    """Core verification module for mathematical expressions."""
    
    def __init__(self):
        self.gold_extraction_config = [ExprExtractionConfig()]
        self.pred_extraction_config = [LatexExtractionConfig(), ExprExtractionConfig()]
        self.float_rounding = 6
        self.numeric_precision = 15
        self.strict = True
    
    def parse_expression(self, expression: str, is_gold: bool = False):
        """Parse a mathematical expression."""
        config = self.gold_extraction_config if is_gold else self.pred_extraction_config
        try:
            parsed = parse(expression, extraction_config=config)
            return parsed if parsed else []
        except Exception as e:
            warnings.warn(f"Error parsing expression '{expression}': {e}")
            return []
    
    def verify_answer(self, gold: str, prediction: str, return_details: bool = False):
        """Verify if a prediction matches the gold answer."""
        try:
            gold_parsed = self.parse_expression(gold, is_gold=True)
            pred_parsed = self.parse_expression(prediction, is_gold=False)
            
            if not gold_parsed or not pred_parsed:
                if return_details:
                    return {
                        'valid': False,
                        'gold': gold,
                        'prediction': prediction,
                        'error_type': 'Parse Error',
                        'details': f"Could not parse: gold={bool(gold_parsed)}, pred={bool(pred_parsed)}"
                    }
                return False
            
            gold_expr = gold_parsed[0] if isinstance(gold_parsed, list) else gold_parsed
            pred_expr = pred_parsed[0] if isinstance(pred_parsed, list) else pred_parsed
            
            is_valid = verify(
                gold_expr,
                pred_expr,
                float_rounding=self.float_rounding,
                numeric_precision=self.numeric_precision,
                strict=self.strict
            )
            
            if return_details:
                return {
                    'valid': is_valid,
                    'gold': gold,
                    'prediction': prediction,
                    'gold_parsed': str(gold_expr),
                    'pred_parsed': str(pred_expr),
                    'error_type': None if is_valid else 'Mathematical Mismatch',
                    'details': 'Expressions are equivalent' if is_valid else 'Expressions are not equivalent'
                }
            
            return is_valid
        except Exception as e:
            warnings.warn(f"Error verifying answer: {e}")
            if return_details:
                return {
                    'valid': False,
                    'gold': gold,
                    'prediction': prediction,
                    'error_type': f'Error: {str(e)}',
                    'details': str(e)
                }
            return False

# Initialize verifier
verifier = MathVerifier()
print("✅ MathVerifier initialized!")


In [ ]:
# Test verification
result = verifier.verify_answer("1/2", "0.5", return_details=True)
print("Test Result:")
print(f"  Gold: {result['gold']}")
print(f"  Prediction: {result['prediction']}")
print(f"  Valid: {result['valid']}")
print(f"  Details: {result['details']}")

if result['valid']:
    print("\n✅ System is working correctly!")
else:
    print("\n⚠️  System working but test case needs different format")


## Step 5: Launch Interactive Web Interface

This will create a Gradio interface that works in Colab!


In [ ]:
import gradio as gr
from IPython.display import HTML, display

def verify_math(gold, pred):
    """Verify mathematical answer with detailed output."""
    result = verifier.verify_answer(gold, pred, return_details=True)
    
    if result['valid']:
        status = "✅ CORRECT"
        color = "green"
    else:
        status = "❌ INCORRECT"
        color = "red"
    
    output = f"""
    <div style="font-family: Arial, sans-serif; padding: 20px;">
        <h2 style="color: {color};">{status}</h2>
        <p><b>Gold Answer:</b> <code>{result['gold']}</code></p>
        <p><b>Prediction:</b> <code>{result['prediction']}</code></p>
        <p><b>Details:</b> {result['details']}</p>
    """
    
    if result.get('gold_parsed'):
        output += f"<p><b>Gold Parsed:</b> <code>{result['gold_parsed']}</code></p>"
    if result.get('pred_parsed'):
        output += f"<p><b>Prediction Parsed:</b> <code>{result['pred_parsed']}</code></p>"
    if result.get('error_type'):
        output += f"<p><b>Error Type:</b> {result['error_type']}</p>"
    
    output += "</div>"
    return output

# Create Gradio interface
iface = gr.Interface(
    fn=verify_math,
    inputs=[
        gr.Textbox(label="Gold/Expected Answer", placeholder="e.g., 1/2 or $\\frac{1}{2}$"),
        gr.Textbox(label="Model Prediction", placeholder="e.g., 0.5 or $\\frac{1}{2}$")
    ],
    outputs=gr.HTML(label="Verification Result"),
    title="🔬 MathVerify: Mathematical Reasoning Verification",
    description="Enter a gold (correct) answer and a model prediction to verify if they match.",
    examples=[
        ["1/2", "0.5"],
        ["$\\frac{1}{2}$", "0.5"],
        ["42", "43"],
        ["$\\sqrt{4}$", "2"],
    ]
)

# Launch with public URL for Colab
print("🚀 Launching interface...")
print("The interface will open below with a public URL!")
print()

iface.launch(share=True, debug=True)


## Step 6: Use Python API (Alternative)

You can also use the verification system programmatically:


In [ ]:
# Example: Verify multiple answers
test_cases = [
    {"gold": "1/2", "pred": "0.5"},
    {"gold": "2+2", "pred": "4"},
    {"gold": "42", "pred": "43"},
]

print("Batch Verification Results:")
print("=" * 50)

for i, case in enumerate(test_cases, 1):
    result = verifier.verify_answer(case["gold"], case["pred"], return_details=True)
    status = "✅" if result['valid'] else "❌"
    print(f"{i}. {status} Gold: {case['gold']} | Pred: {case['pred']} | Valid: {result['valid']}")

print("\n✅ API is working!")
